# Лабораторная работа №4

Реализация основных моделей в агентном подходе. Модель SIR

Закиров Нурислам Дамирович (РУДН)

В данной работе реализуется агентная модель распространения
инфекционного заболевания на основе классической компартментальной
модели SIR (Susceptible-Infectious-Recovered). Модель реализована с
использованием пакета Agents.jl на языке Julia.

# Введение

Модель SIR, предложенная Кермаком и Маккендриком в 1927 году, описывает
динамику эпидемии в популяции, разделённой на три группы:

-   **S (Susceptible)** — восприимчивые к заболеванию индивиды;
-   **I (Infectious)** — инфицированные, способные заражать
    восприимчивых;
-   **R (Recovered)** — выздоровевшие (или умершие), получившие
    иммунитет.

## Классическая модель SIR

Классическая модель описывается системой обыкновенных дифференциальных
уравнений:

$$
\frac{dS}{dt} = -\beta SI/N, \quad \frac{dI}{dt} = \beta SI/N - \gamma I, \quad \frac{dR}{dt} = \gamma I
$$

где $\beta$ — коэффициент передачи инфекции, $\gamma$ — скорость
выздоровления.

## Преимущества агентного подхода

Агентное моделирование позволяет преодолеть ограничения классического
подхода:

-   Каждый индивид моделируется отдельно с уникальными характеристиками
-   Взаимодействия происходят локально в пространстве или социальной
    сети
-   Процессы носят стохастический характер
-   Можно учитывать гетерогенность контактов, мобильность, меры контроля

# Инициализация проекта и загрузка пакетов

In [1]:
using DrWatson
@quickactivate "project"
using Agents, DataFrames, Plots
using JLD2
include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

# Параметры эксперимента

In [2]:
# Параметры эксперимента
params = Dict(
    :Ns => [1000, 1000, 1000],
    :β_und => [0.5, 0.5, 0.5],
    :β_det => [0.05, 0.05, 0.05],
    :infection_period => 14,
    :detection_time => 7,
    :death_rate => 0.02,
    :reinfection_probability => 0.1,
    :Is => [0, 0, 1],
    :seed => 42,
    :n_steps => 100,
)

Dict{Symbol, Any} with 10 entries:
  :Is                      => [0, 0, 1]
  :seed                    => 42
  :infection_period        => 14
  :n_steps                 => 100
  :β_und                   => [0.5, 0.5, 0.5]
  :Ns                      => [1000, 1000, 1000]
  :detection_time          => 7
  :reinfection_probability => 0.1
  :β_det                   => [0.05, 0.05, 0.05]
  :death_rate              => 0.02

# Инициализация модели

In [3]:
# Инициализация модели
model = initialize_sir(; params...)

StandardABM with 3000 agents of type Person
 agents container: Dict
 space: GraphSpace with 3 positions and 3 edges
 scheduler: fastest
 properties: C, infection_period, β_und, Ns, migration_rates, detection_time, reinfection_probability, β_det, death_rate

# Запуск симуляции

In [4]:
# Подготовка массивов для хранения данных
times = Int[]
S_vals = Int[]
I_vals = Int[]
R_vals = Int[]
total_vals = Int[]

# Запуск симуляции вручную
for step = 1:params[:n_steps]
    Agents.step!(model, 1)
    push!(times, step)
    push!(S_vals, susceptible_count(model))
    push!(I_vals, infected_count(model))
    push!(R_vals, recovered_count(model))
    push!(total_vals, total_count(model))
end

# Создание DataFrame

In [5]:
# Создаём DataFrame для удобства
agent_df = DataFrame(
    time = times,
    susceptible = S_vals,
    infected = I_vals,
    recovered = R_vals
)
model_df = DataFrame(time = times, total = total_vals)

# Визуализация результатов

In [6]:
# Визуализация
plot(
    agent_df.time,
    agent_df.susceptible,
    label = "Восприимчивые",
    xlabel = "Дни",
    ylabel = "Количество",
)
plot!(agent_df.time, agent_df.infected, label = "Инфицированные")
plot!(agent_df.time, agent_df.recovered, label = "Выздоровевшие")
plot!(agent_df.time, model_df.total, label = "Всего (включая умерших)", linestyle = :dash)
savefig(plotsdir("sir_basic_dynamics.png"))

"/home/nurislam/rudn1/labs/lab04/project/plots/sir_basic_dynamics.png"

# Сохранение данных

In [7]:
# Сохранение данных
@save datadir("sir_basic_agent.jld2") agent_df
@save datadir("sir_basic_model.jld2") model_df

# Выводы

В ходе данной работы была реализована агентная модель SIR. Базовый
эксперимент показал:

1.  Эпидемия развивается с характерным пиком инфицированных
2.  Часть населения выздоравливает и приобретает иммунитет
3.  Наблюдается небольшая смертность (2% от заболевших)

# Литература

1.  Kermack W. O., McKendrick A. G. A Contribution to the Mathematical
    Theory of Epidemics // Proceedings of the Royal Society of London.
    Series A. — 1927.
2.  Hethcote H. W. The Mathematics of Infectious Diseases // SIAM
    Review. — 2000.
3.  Agents.jl: Agent-Based Modeling in Julia. — 2024.